# 11 - Query plans, skew, shuffle tuning, and table layout

Prerequisites / execution context: Run 00. Small bounded dataset. Plan inspection works broadly; detailed Spark UI and cluster setting exercises require compatible classic compute. No benchmark claims from tiny data.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R18: PySpark overview](https://docs.databricks.com/aws/en/pyspark/); [R52: Spark UI diagnosis](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/); [R53: Skew and spill](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/long-spark-stage-page); [R54: Spark memory issues](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/spark-memory-issues); [R55: Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe); [R56: Liquid clustering](https://docs.databricks.com/aws/en/tables/clustering); [R57: Predictive optimization](https://docs.databricks.com/aws/en/optimizations/predictive-optimization); [R04: Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Build a skewed distribution
Most events have the same key. Inspect the plan and per-key distribution before changing settings.


In [ ]:
fact=(spark.range(100000).withColumn("k",F.when(F.col("id")<90000,F.lit(0)).otherwise(F.col("id")%100))
      .withColumn("amount",F.lit(1)))
dim=spark.range(100).select(F.col("id").alias("k"),F.concat(F.lit("group_"),F.col("id")).alias("label"))
fact.groupBy("k").count().orderBy(F.desc("count")).show(5)
joined=fact.join(F.broadcast(dim),"k")
joined.explain("formatted")
assert joined.count()==100000


## 2. Verify a salted aggregation preserves the answer
Partial sums distribute a hot key; combine partial sums by the original key. This applies to a decomposable SUM. Averages need sum and count; exact distinct needs different reasoning. AQE skew-join handling is not a universal fix for skewed aggregation. Official Databricks documentation: [R53: Skew and spill](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/long-spark-stage-page); [R55: Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe).


In [ ]:
plain=fact.groupBy("k").agg(F.sum("amount").alias("total"))
salted=(fact.withColumn("salt",(F.col("id")%16).cast("int"))
 .groupBy("k","salt").agg(F.sum("amount").alias("partial"))
 .groupBy("k").agg(F.sum("partial").alias("total")))
assert plain.exceptAll(salted).count()==0
assert salted.exceptAll(plain).count()==0
salted.explain("formatted")


## 3. Measure one change at a time
On compatible classic compute: capture baseline runtime, input size, shuffle read/write, spill, and max/median task duration. Compare shuffle partition settings; restore the original setting afterward. More partitions reduce work per task but add overhead. Memory and default parallelism are not interchangeable with SQL shuffle partitions. On serverless, many settings cannot be configured and Spark UI access differs; use available query profiles. Official Databricks documentation: [R04: Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations); [R52: Spark UI diagnosis](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/); [R54: Spark memory issues](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/spark-memory-issues); [R55: Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe).


In [ ]:
RUN_CLASSIC_TUNING=False
if RUN_CLASSIC_TUNING:
    import time
    key="spark.sql.shuffle.partitions"
    old=spark.conf.get(key)
    measurements=[]
    try:
        for setting in ("8","32"):
            spark.conf.set(key,setting)
            start=time.perf_counter()
            fact.groupBy("k").sum("amount").collect()  # bounded 100-group result
            measurements.append((setting,time.perf_counter()-start))
    finally:
        spark.conf.set(key,old)
    print(measurements)


## 4. Liquid clustering sandbox
This optional operation requires supported table/runtime features; skip if unavailable. Tiny data cannot demonstrate pruning benefits. For a meaningful comparison use representative production-shaped data in a separate lab. Official Databricks documentation: [R56: Liquid clustering](https://docs.databricks.com/aws/en/tables/clustering).


In [ ]:
RUN_CLUSTERING=False
if RUN_CLUSTERING:
    T=f"{SILVER}.cluster_demo"
    spark.sql(f"CREATE OR REPLACE TABLE {T} (id BIGINT,k BIGINT) USING DELTA CLUSTER BY (k)")
    fact.select("id","k").write.mode("append").saveAsTable(T)
    spark.sql(f"OPTIMIZE {T}")
    spark.sql(f"DESCRIBE DETAIL {T}").show(truncate=False)


## Diagnostic cases
A few slow huge tasks: inspect skew. Many large spilling tasks: inspect partition sizing and memory per core. Driver OOM after collect: reduce driver collection. Failure before first task: inspect compute events, permissions, networking, or library environment. Predictive optimization automates maintenance for eligible UC managed tables; it does not repair an incorrect join. Official Databricks documentation: [R52: Spark UI diagnosis](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/); [R54: Spark memory issues](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/spark-memory-issues); [R57: Predictive optimization](https://docs.databricks.com/aws/en/optimizations/predictive-optimization).
